# NSE Feature Ablation (Phase 2, Tier A) — does microstructure information help?

Tests whether engineered microstructure channels improve NSE index-direction prediction over the raw
40-feature LOB window. We sweep **feature sets** for our proposed **MambaLOB** (and **TLOB** as a
SOTA check) — both are feature-count-agnostic via a linear input projection (DeepLOB's convs assume the
4-per-level layout, so it's excluded from the non-40/80 sets).

Feature sets (see `modeling/features.py`):
**base40** (raw LOB) · **micro** (+OFI, depth imbalance, micro-price, rel-spread = 46) ·
**orders60** (+per-level order counts, *Dhan-unique* = 60) · **depth80** (20 levels = 80) ·
**all** (base40+micro+orders = 66).

Needs GPU + secrets `GH_PAT`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`.


## 1. Runtime check

In [ ]:
import torch, platform
print("Python:", platform.python_version(), "| Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Get the project code (GH_PAT secret)

In [ ]:
import sys, subprocess, pathlib
REPO_URL = "https://github.com/rajjoseph48/nse-lob-capstone.git"
REPO_DIR = "nse-lob-capstone"
def _get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    import os
    return os.environ.get(name, "")
GITHUB_TOKEN = _get_secret("GH_PAT")
def add_modeling(base):
    base = pathlib.Path(base)
    for cand in (base / "modeling", base):
        if (cand / "models.py").exists():
            sys.path.insert(0, str(cand.resolve()))
            return str(cand.resolve())
    return None
path = None
for c in (".", REPO_DIR, "/kaggle/working/" + REPO_DIR, "/content/" + REPO_DIR):
    path = add_modeling(c)
    if path:
        break
if not path:
    url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url], check=True)
    path = add_modeling(REPO_DIR)
print("modeling/ on sys.path:", path)

## 3. Install deps (Mamba kernel + boto3)

In [ ]:
import subprocess, sys
def pipq(*pkgs): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pipq("boto3")
pipq("ninja", "packaging", "setuptools", "wheel")
pipq("--no-build-isolation", "causal-conv1d")
pipq("--no-build-isolation", "mamba-ssm")
try:
    import mamba_ssm
    print("install step done | mamba-ssm", mamba_ssm.__version__)
except Exception as e:
    print("install step done | mamba-ssm NOT importable -> pure-PyTorch fallback:", repr(e))

## 4. Download NSE Dhan data from S3 (AWS_* secrets)

In [ ]:
import os, boto3, pathlib
os.environ["AWS_ACCESS_KEY_ID"] = _get_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = _get_secret("AWS_SECRET_ACCESS_KEY")
BUCKET, PREFIX, REGION = "lob-capstone-data", "lob-data/dhan/", "ap-south-2"
DATA_DIR_NSE = "nse_data/dhan"; pathlib.Path(DATA_DIR_NSE).mkdir(parents=True, exist_ok=True)
s3 = boto3.client("s3", region_name=REGION)
n = 0
for o in s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX).get("Contents", []):
    if o["Size"] == 0:
        continue
    dst = os.path.join(DATA_DIR_NSE, o["Key"].split("/")[-1])
    if not os.path.exists(dst):
        s3.download_file(BUCKET, o["Key"], dst)
    n += 1
print(f"{n} parquet files in {DATA_DIR_NSE}")

## 5. Metrics + baselines

In [ ]:
import numpy as np, torch
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             matthews_corrcoef, confusion_matrix)
from fi2010_dataset import make_loader
from train import DEVICE
CLASS_NAMES = ["Down", "Stat", "Up"]
def collect_preds(model, ds, batch_size=256):
    model = model.to(DEVICE).eval()
    loader = make_loader(ds, batch_size=batch_size, shuffle=False)
    preds, labels = [], []
    with torch.no_grad():
        for X, y in loader:
            preds.append(model(X.to(DEVICE)).argmax(1).cpu().numpy()); labels.append(y.numpy())
    return np.concatenate(preds), np.concatenate(labels)
def full_metrics(y_true, y_pred):
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
            "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
            "mcc": float(matthews_corrcoef(y_true, y_pred))}
def naive_baselines(y_train, y_test):
    maj = int(np.bincount(y_train, minlength=3).argmax())
    wf1 = lambda yp: f1_score(y_test, yp, average="weighted", zero_division=0)
    rng = np.random.default_rng(0); probs = np.bincount(y_train, minlength=3) / len(y_train)
    return {"baseline_majority_wf1": round(wf1(np.full_like(y_test, maj)), 4),
            "baseline_stat_wf1": round(wf1(np.full_like(y_test, 1)), 4),
            "baseline_random_wf1": round(wf1(rng.choice(3, size=len(y_test), p=probs)), 4)}

## 6. Ablation runner
Builds the model with `n_features` taken from the loaded feature set (so any set works). Per-run S3 sync
to `feature_ablation.csv`; resumable.

In [ ]:
import time, json, pathlib, pandas as pd
from nse_dataset import load_nse
from models import build_model
from train import train, save_checkpoint
from stats import set_seed
from nbenv import s3_client, s3_put, s3_pull

RESULTS_DIR = pathlib.Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
CKPT_DIR = pathlib.Path("checkpoints/featabl"); CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RESULTS_DIR / "feature_ablation.csv"
MODEL_LR = {"mambalob": 3e-4, "tlob": 1e-4, "mlplob": 1e-3}
S3_BUCKET, S3_PREFIX = "lob-capstone-data", "reproduction/featabl"
_s3 = s3_client(); s3_pull(_s3, RESULTS_CSV, S3_BUCKET, S3_PREFIX)

def run_ablation(model_name, feature_set, horizon, symbol="NIFTY", seed=0,
                 epochs=20, patience=3, batch_size=128):
    set_seed(seed)
    tag = f"{model_name}_{symbol}_{feature_set}_h{horizon}_s{seed}"
    print("=" * 74); print(" ", tag); print("=" * 74)
    tr, vl, te = load_nse(symbol=symbol, horizon=horizon, seq_len=100,
                          feature_set=feature_set, data_dir=DATA_DIR_NSE)
    F = tr[0][0].shape[-1]
    model = build_model(model_name, seq_len=100, n_features=F)
    lr = MODEL_LR.get(model_name, 1e-3)
    t0 = time.time()
    hist = train(model, tr, vl, epochs=epochs, patience=patience, batch_size=batch_size, lr=lr, verbose=True)
    elapsed = time.time() - t0
    y_pred, y_true = collect_preds(model, te); mt = full_metrics(y_true, y_pred)
    bl = naive_baselines(tr.y.numpy(), y_true)
    ckpt = CKPT_DIR / f"{tag}.pt"; save_checkpoint(model, str(ckpt))
    row = {"model": model_name, "symbol": symbol, "feature_set": feature_set, "n_features": F,
           "horizon": horizon, "seed": seed, "n_params": sum(p.numel() for p in model.parameters()),
           "test_accuracy": round(mt["accuracy"], 4), "test_macro_f1": round(mt["macro_f1"], 4),
           "test_weighted_f1": round(mt["weighted_f1"], 4), "test_mcc": round(mt["mcc"], 4),
           **bl, "train_time_s": round(elapsed, 1)}
    df = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True); df.to_csv(RESULTS_CSV, index=False)
    for _p in (RESULTS_CSV, ckpt):
        s3_put(_s3, _p, S3_BUCKET, S3_PREFIX)
    print(f"  -> wF1={mt['weighted_f1']:.4f} mF1={mt['macro_f1']:.4f} (F={F}, {elapsed:.0f}s)")
    return row

def _done():
    if not RESULTS_CSV.exists(): return set()
    d = pd.read_csv(RESULTS_CSV)
    return {(r.model, r.symbol, r.feature_set, int(r.horizon)) for r in d.itertuples()}

## 7. Run the ablation
Default: **MambaLOB** (proposed) over all feature sets on NIFTY at k=10 and k=100 — the core question.
Then a TLOB spot-check (base40 vs micro). Edit the lists to widen. Resumable.

In [ ]:
FEATURE_SETS = ["base40", "micro", "orders60", "depth80", "all"]
HORIZONS = [10, 100]
done = _done()

# Proposed model across all feature sets
for fs in FEATURE_SETS:
    for h in HORIZONS:
        if ("mambalob", "NIFTY", fs, h) in done:
            print(f"skip mambalob {fs} h{h}"); continue
        run_ablation("mambalob", fs, h, symbol="NIFTY")

# SOTA spot-check: do the features also help TLOB?
for fs in ["base40", "micro"]:
    if ("tlob", "NIFTY", fs, 10) in _done():
        print(f"skip tlob {fs} h10"); continue
    run_ablation("tlob", fs, 10, symbol="NIFTY")

pd.read_csv(RESULTS_CSV)

## 8. Findings — does microstructure help?

In [ ]:
import matplotlib.pyplot as plt
res = pd.read_csv(RESULTS_CSV)
ORDER = ["base40", "micro", "orders60", "depth80", "all"]

mam = res[res.model == "mambalob"].copy()
piv = mam.pivot_table(index="feature_set", columns="horizon", values="test_weighted_f1").reindex(ORDER)
print("MambaLOB (NIFTY) weighted-F1 by feature set x horizon:")
print(piv.round(4).to_string())
# delta vs base40
delta = (piv - piv.loc["base40"]).round(4)
print("\nDelta vs base40 (positive = features help):")
print(delta.to_string())

ax = piv.plot(marker="o"); ax.set(title="MambaLOB NIFTY — weighted-F1 by feature set",
    xlabel="feature set", ylabel="weighted-F1"); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "fig_feature_ablation.png", dpi=150, bbox_inches="tight")
s3_put(_s3, RESULTS_DIR / "fig_feature_ablation.png", S3_BUCKET, S3_PREFIX); plt.show()

if (res.model == "tlob").any():
    print("\nTLOB spot-check (NIFTY, k=10):")
    print(res[res.model == "tlob"][["feature_set", "test_weighted_f1", "test_macro_f1"]].to_string(index=False))